# Lab 05 — Autoencoders and latent representations

**Dataset:** [ylecun/mnist](https://huggingface.co/datasets/ylecun/mnist). Train a bottleneck autoencoder, then test whether denoising improves robustness to corrupted inputs.

## The math behind the experiment

An autoencoder has an encoder and decoder:

$$
z=f_\phi(x),\qquad \hat{x}=g_\theta(z).
$$

Training minimizes reconstruction error, for example

$$
L_{rec}=\frac{1}{N}\sum_{i=1}^{N}\|x_i-\hat{x}_i\|_2^2.
$$

The bottleneck $z$ forces the network to decide what information is worth preserving. A denoising autoencoder makes this more interesting by encoding a corrupted input $\tilde{x}$ while still reconstructing the clean $x$.

In [ ]:
!pip -q install datasets torch pandas
from datasets import load_dataset
import numpy as np,pandas as pd,torch
from torch import nn
tr=load_dataset("ylecun/mnist",split="train[:3000]"); te=load_dataset("ylecun/mnist",split="test[:500]")
def arr(ds): return torch.tensor(np.array([np.array(x).reshape(-1) for x in ds["image"]])/255.,dtype=torch.float32)
Xtr,Xte=arr(tr),arr(te)
class AE(nn.Module):
 def __init__(self,z=16): super().__init__(); self.enc=nn.Sequential(nn.Linear(784,128),nn.ReLU(),nn.Linear(128,z)); self.dec=nn.Sequential(nn.Linear(z,128),nn.ReLU(),nn.Linear(128,784),nn.Sigmoid())
 def forward(self,x): return self.dec(self.enc(x))
def train_ae(noise=0.,z=16):
 m=AE(z); opt=torch.optim.Adam(m.parameters(),1e-3); loss=nn.MSELoss()
 for _ in range(8):
  xb=Xtr+noise*torch.randn_like(Xtr); xb=xb.clamp(0,1); opt.zero_grad(); loss(m(xb),Xtr).backward(); opt.step()
 with torch.no_grad(): rec=m(Xte); mse=loss(rec,Xte).item(); return m,mse
base,base_mse=train_ae(0.); print("clean reconstruction MSE",base_mse)

## Step 1 — Prediction

A smaller latent bottleneck should force compression and increase reconstruction error. A denoising objective should make the decoder reconstruct clean images from corrupted inputs, so it may outperform the plain autoencoder on noisy test inputs.

In [ ]:
denoise,denoise_clean=train_ae(.30); noise=.30*torch.randn_like(Xte); noisy=(Xte+noise).clamp(0,1)
loss=nn.MSELoss()
with torch.no_grad():
 plain_noisy=loss(base(noisy),Xte).item(); denoise_noisy=loss(denoise(noisy),Xte).item()
rows=[["plain",base_mse,plain_noisy],["denoising",denoise_clean,denoise_noisy)]
pd.DataFrame(rows,columns=["model","clean_MSE","noisy_input_MSE"])

## Step 2 — Bottleneck intervention

Change only latent size. Record reconstruction error and parameter count.

In [ ]:
rows=[]
for z in [4,16,64]:
 m,mse=train_ae(0.,z); rows.append([z,sum(p.numel() for p in m.parameters()),mse])
pd.DataFrame(rows,columns=["latent_dim","parameters","test_reconstruction_MSE"])

## Visualize the representation — reconstruction and bottleneck size

First compare how the bottleneck dimension changes held-out reconstruction error. Then inspect the same test images as input, noisy input, and reconstructions. The picture helps identify whether the model preserves recognizable structure or merely optimizes average pixel error.

In [ ]:
import matplotlib.pyplot as plt

latent_results = pd.DataFrame(rows, columns=["latent_dim", "parameters", "test_reconstruction_MSE"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(latent_results["latent_dim"], latent_results["test_reconstruction_MSE"], marker="o")
axes[0].set_title("Bottleneck size vs reconstruction error")
axes[0].set_xlabel("Latent dimension")
axes[0].set_ylabel("Test reconstruction MSE")
axes[0].grid(alpha=.25)

with torch.no_grad():
    clean = Xte[:6]
    noisy_sample = (clean + .30 * torch.randn_like(clean)).clamp(0, 1)
    plain_rec = base(noisy_sample)
    denoise_rec = denoise(noisy_sample)
images = [clean, noisy_sample, plain_rec, denoise_rec]
titles = ["Clean target", "Noisy input", "Plain AE", "Denoising AE"]
axes[1].axis("off")
fig2, axs = plt.subplots(4, 6, figsize=(10, 6))
for r, (batch, title) in enumerate(zip(images, titles)):
    for c in range(6):
        axs[r, c].imshow(batch[c].reshape(28, 28).cpu().numpy(), cmap="gray", vmin=0, vmax=1)
        axs[r, c].axis("off")
    axs[r, 0].set_ylabel(title, rotation=0, labelpad=55, va="center")
fig2.suptitle("What does the autoencoder reconstruct?")
fig2.tight_layout()
plt.show()
fig.tight_layout()
plt.show()

## Conclusion / answer key

Expected: smaller bottlenecks usually increase reconstruction error; denoising training should reduce sensitivity to input corruption when the noise level matches the training intervention. A low reconstruction error does not automatically mean the latent space is useful for every downstream task.

### Research extension
Freeze encoder representations and test a linear classifier. Compare representation quality against reconstruction quality.